# Step 1: The Setup & South African Demographics

In [2]:
import numpy as np
import pandas as pd

# 1. Set a random seed so your results are reproducible every time you run this
np.random.seed(42)
n_samples = 10000

print(f"Initializing engine to generate {n_samples:,} South African applicant profiles...\n")

# 2. Generate Unique Client IDs (e.g., CAP00001, CAP00002...)
client_ids = [f'CAP{i:05d}' for i in range(1, n_samples + 1)]

# 3. Age: Standard adult credit-seeking age (18 to 70 years old)
age = np.random.randint(18, 70, size=n_samples)

# 4. Gender: Roughly 49% Male, 51% Female
gender = np.random.choice(['Male', 'Female'], size=n_samples, p=[0.49, 0.51])

# 5. Population Groups: Weighted using approximate SA Census distributions
# This will allow us to audit our AI model for fairness/bias later!
pop_groups = ['Black', 'Coloured', 'Indian', 'White']
pop_weights = [0.81, 0.09, 0.026, 0.074]  # Must sum to exactly 1.0
population_group = np.random.choice(pop_groups, size=n_samples, p=pop_weights)

# 6. Provinces: Weighted by approximate economic activity/population density
provinces = ['Gauteng', 'KwaZulu-Natal', 'Western Cape', 'Eastern Cape', 'Limpopo', 'Mpumalanga', 'North West', 'Free State', 'Northern Cape']
prov_weights = [0.26, 0.19, 0.12, 0.11, 0.10, 0.08, 0.07, 0.05, 0.02]  # Must sum to 1.0
province = np.random.choice(provinces, size=n_samples, p=prov_weights)

# 7. Combine these initial arrays into our starter Pandas DataFrame
df = pd.DataFrame({
    'Client_ID': client_ids,
    'Age': age,
    'Gender': gender,
    'Province': province,
    'Population_Group': population_group
})

# Let's inspect the first 5 rows to see our starter data
print(df.head())

Initializing engine to generate 10,000 South African applicant profiles...

  Client_ID  Age  Gender      Province Population_Group
0  CAP00001   56  Female       Gauteng            White
1  CAP00002   69    Male    Mpumalanga            Black
2  CAP00003   46    Male  Eastern Cape            Black
3  CAP00004   32  Female  Eastern Cape            Black
4  CAP00005   60    Male    North West            Black


# Step 2: Income and SASSA Grants!

In [3]:
# =====================================================================
# STEP 2: EMPLOYMENT, INCOME, AND SASSA GRANTS
# =====================================================================
print("Generating realistic income profiles and SASSA statuses...\n")

# 1. Employment Types (Capitec's retail base includes informal and self-employed workers)
emp_types = ['Formal Sector', 'Informal Sector', 'Self-Employed', 'Unemployed']
emp_weights = [0.60, 0.15, 0.10, 0.15]  # 15% unemployment in this applicant pool
employment_type = np.random.choice(emp_types, size=n_samples, p=emp_weights)

# 2. Base Income using a Log-Normal Distribution (simulating SA's income spread)
# mean=9.2 and sigma=0.8 centers our median income around R10,000
income_base = np.random.lognormal(mean=9.2, sigma=0.8, size=n_samples)
monthly_income = np.clip(income_base, 2500, 180000)  # Caps incomes between R2,500 and R180,000

# 3. Logic Check: Adjust income for Unemployed applicants
# If unemployed, their income should be low (simulating odd jobs or pocket money between R350 and R3,500)
for i in range(n_samples):
    if employment_type[i] == 'Unemployed':
        monthly_income[i] = np.random.uniform(350, 3500)

# 4. SASSA Grant Recipient Logic
# We loop through our incomes and assign grants based on how much a person earns
sassa_grant = []
for inc in monthly_income:
    if inc < 3500:
        # High probability (85%) of receiving a grant if income is under R3,500
        sassa_grant.append(np.random.choice(['Yes', 'No'], p=[0.85, 0.15]))
    elif inc < 8000:
        # Lower probability (30%) if income is between R3,500 and R8,000
        sassa_grant.append(np.random.choice(['Yes', 'No'], p=[0.30, 0.70]))
    else:
        # Zero chance if income is above R8,000
        sassa_grant.append('No')

# Convert our Python list into a fast NumPy array
sassa_grant = np.array(sassa_grant)

# 5. Append these new features to our existing DataFrame
df['Employment_Type'] = employment_type
df['Monthly_Income_ZAR'] = np.round(monthly_income, 2)
df['SASSA_Grant_Recipient'] = sassa_grant

# Let's see how our simulated South Africans are looking now!
print(df.head(10))

Generating realistic income profiles and SASSA statuses...

  Client_ID  Age  Gender       Province Population_Group Employment_Type  \
0  CAP00001   56  Female        Gauteng            White   Self-Employed   
1  CAP00002   69    Male     Mpumalanga            Black   Formal Sector   
2  CAP00003   46    Male   Eastern Cape            Black   Formal Sector   
3  CAP00004   32  Female   Eastern Cape            Black      Unemployed   
4  CAP00005   60    Male     North West            Black   Formal Sector   
5  CAP00006   25    Male        Gauteng            Black   Formal Sector   
6  CAP00007   38  Female        Gauteng            Black   Formal Sector   
7  CAP00008   56    Male   Western Cape            Black   Self-Employed   
8  CAP00009   36  Female        Limpopo            Black   Formal Sector   
9  CAP00010   40  Female  KwaZulu-Natal            White      Unemployed   

   Monthly_Income_ZAR SASSA_Grant_Recipient  
0            11517.19                    No  
1          

# Step 3: Credit Bureau Scores and Debt-to-Income (DTI) ratios

In [4]:
# =====================================================================
# STEP 3: CREDIT SCORES AND DEBT-TO-INCOME (DTI) RATIOS
# =====================================================================
print("Calculating TransUnion-scale credit scores and DTI ratios...\n")

# 1. Calculate a Base Credit Score linked to Income
# We use np.log10 so that credit scores taper off as income gets very high
base_score = 550 + 100 * np.log10(df['Monthly_Income_ZAR'] / 2500)

# 2. Modify the Base Score using our Employment Type from Step 2
# We loop through our dataframe indexes using a standard pattern
base_score_array = base_score.to_numpy() # Convert to numpy for raw speed

for i in range(n_samples):
    if df['Employment_Type'].iloc[i] == 'Unemployed':
        base_score_array[i] -= 100  # Penalty for no formal income
    elif df['Employment_Type'].iloc[i] == 'Formal Sector':
        base_score_array[i] += 50   # Boost for steady, verified employment

# 3. Add Normal Distribution "Noise" so scores aren't perfectly predictable
# mean (loc) = 0, standard deviation (scale) = 75 points
noise = np.random.normal(loc=0, scale=75, size=n_samples)
final_credit_score = base_score_array + noise

# Clip the scores to the official South African bureau bounds (300 to 850)
df['Credit_Bureau_Score'] = np.clip(final_credit_score, 300, 850).astype(int)

# 4. Generate Debt-to-Income (DTI) using a Beta Distribution
# a=2, b=5 shapes the curve so the average DTI sits around 0.28 (28%)
base_dti = np.random.beta(a=2, b=5, size=n_samples)

# 5. Logic Check: Adjust DTI based on Credit Score
# If your credit score is terrible (< 450), banks won't lend to you, so your DTI drops.
# If your credit score is excellent (> 700), you have high access to vehicle finance/bonds.
for i in range(n_samples):
    if df['Credit_Bureau_Score'].iloc[i] < 450:
        base_dti[i] = base_dti[i] * 0.4  # Restrained access to debt
    elif df['Credit_Bureau_Score'].iloc[i] > 700:
        base_dti[i] = base_dti[i] * 1.2  # High utilization of credit facilities

df['Debt_to_Income_Ratio'] = np.round(base_dti, 4)

# Let's check out our updated dataset
print(df[['Client_ID', 'Employment_Type', 'Monthly_Income_ZAR', 'Credit_Bureau_Score', 'Debt_to_Income_Ratio']].head(10))

Calculating TransUnion-scale credit scores and DTI ratios...

  Client_ID Employment_Type  Monthly_Income_ZAR  Credit_Bureau_Score  \
0  CAP00001   Self-Employed            11517.19                  630   
1  CAP00002   Formal Sector            16214.92                  711   
2  CAP00003   Formal Sector            18688.00                  792   
3  CAP00004      Unemployed             3026.21                  451   
4  CAP00005   Formal Sector             5172.87                  658   
5  CAP00006   Formal Sector             7871.35                  654   
6  CAP00007   Formal Sector             3098.67                  590   
7  CAP00008   Self-Employed             7794.73                  583   
8  CAP00009   Formal Sector            12810.43                  612   
9  CAP00010      Unemployed              911.08                  397   

   Debt_to_Income_Ratio  
0                0.3458  
1                0.0962  
2                0.2841  
3                0.1333  
4              

# Step 4: Building the actual mathematical target logic to decide who defaults

In [6]:
# =====================================================================
# STEP 4: DESIGNING DEFAULT LOGIC & NPL METRICS
# =====================================================================
print("Running default probability simulations and calculating NPL metrics...\n")

# 1. Standardize our continuous features (creating mathematical Z-scores)
norm_dti = (df['Debt_to_Income_Ratio'] - 0.25) / 0.15
norm_credit = (df['Credit_Bureau_Score'] - 600) / 100
norm_income = (np.log(df['Monthly_Income_ZAR']) - 9.5) / 1.0

# 2. Calculate the baseline log-odds using our regression coefficients
# We convert our pandas columns to numpy arrays behind the scenes for clean math math operations
log_odds = -3.5 + 1.2 * norm_dti.to_numpy() - 2.0 * norm_credit.to_numpy() - 0.4 * norm_income.to_numpy()

# 3. Apply institutional risk adjustments for specific employment types and grants
for i in range(n_samples):
    if df['Employment_Type'].iloc[i] == 'Unemployed':
        log_odds[i] += 0.8  # Unemployed applicants carry an inherent risk premium
    if df['SASSA_Grant_Recipient'].iloc[i] == 'Yes':
        log_odds[i] -= 0.1  # SASSA grant backing provides a minimal, secure payment floor

# 4. Transform log-odds into a clear probability percentage (0.0 to 1.0)
prob_default = 1 / (1 + np.exp(-log_odds))

# 5. Execute a Bernoulli/Binomial trial to determine final outcome
# np.random.binomial(n=1, p) flips a biased coin with probability 'p'. 
# Returns 1 for default, 0 for successful repayment.
df['Default_Status'] = np.random.binomial(n=1, p=prob_default)

# 6. Calculate our Portfolio-wide Non-Performing Loan (NPL) ratio
npl_ratio = df['Default_Status'].mean()

print(f"--- Portfolio Performance Summary ---")
print(f"Total Simulated Loan Portfolio: {len(df):,} accounts")
print(f"Portfolio Non-Performing Loan (NPL) Ratio: {npl_ratio:.2%}\n")

# Let's inspect the clients who actually defaulted to see if the logic held up
print("Sample of accounts flagged as Non-Performing Loans (Default_Status = 1):")
print(df[df['Default_Status'] == 1][['Client_ID', 'Credit_Bureau_Score', 'Debt_to_Income_Ratio', 'Default_Status']].head(5))

Running default probability simulations and calculating NPL metrics...

--- Portfolio Performance Summary ---
Total Simulated Loan Portfolio: 10,000 accounts
Portfolio Non-Performing Loan (NPL) Ratio: 16.96%

Sample of accounts flagged as Non-Performing Loans (Default_Status = 1):
   Client_ID  Credit_Bureau_Score  Debt_to_Income_Ratio  Default_Status
3   CAP00004                  451                0.1333               1
9   CAP00010                  397                0.1420               1
11  CAP00012                  427                0.1256               1
15  CAP00016                  626                0.4168               1
22  CAP00023                  397                0.2044               1


# Step 5: Training our predictive machine learning model and calculating SHAP explainability values

In [ ]:
#pip install shap


INFO: pip is looking at multiple versions of numba to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of pandas to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of scipy to determine which version is compatible with other requirements. This could take a while.
     ---------------------------------------- 0.0/61.0 kB ? eta -:--:--
     ---------------------------------------- 0.0/61.0 kB ? eta -:--:--
     -------------------------- ------------- 41.0/61.0 kB 2.0 MB/s eta 0:00:01
     -------------------------- ------------- 41.0/61.0 kB 2.0 MB/s eta 0:00:01
     -------------------------- ------------- 41.0/61.0 kB 2.0 MB/s eta 0:00:01
     -------------------------- ------------- 41.0/61.0 kB 2.0 MB/s eta 0:00:01
     -------------------------- ------------- 41.0/61.0 kB 2.0 MB/s eta 0:00:01
     -----------------------

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.0 requires FuzzyTM>=0.4.0, which is not installed.
astropy 5.3.4 requires numpy<2,>=1.21, but you have numpy 2.4.6 which is incompatible.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.4.6 which is incompatible.
matplotlib 3.8.0 requires numpy<2,>=1.21, but you have numpy 2.4.6 which is incompatible.
pywavelets 1.5.0 requires numpy<2.0,>=1.22.4, but you have numpy 2.4.6 which is incompatible.
streamlit 1.30.0 requires numpy<2,>=1.19.3, but you have numpy 2.4.6 which is incompatible.
streamlit 1.30.0 requires pandas<3,>=1.3.0, but you have pandas 3.0.3 which is incompatible.


   - -------------------------------------- 1.2/36.6 MB 80.2 kB/s eta 0:07:22
   - -------------------------------------- 1.2/36.6 MB 80.2 kB/s eta 0:07:22
   - -------------------------------------- 1.2/36.6 MB 81.0 kB/s eta 0:07:18
   - -------------------------------------- 1.2/36.6 MB 81.0 kB/s eta 0:07:18
   - -------------------------------------- 1.2/36.6 MB 81.0 kB/s eta 0:07:18
   - -------------------------------------- 1.2/36.6 MB 81.7 kB/s eta 0:07:14
   - -------------------------------------- 1.2/36.6 MB 82.0 kB/s eta 0:07:12
   - -------------------------------------- 1.2/36.6 MB 82.0 kB/s eta 0:07:12
   - -------------------------------------- 1.3/36.6 MB 82.9 kB/s eta 0:07:07
   - -------------------------------------- 1.3/36.6 MB 82.9 kB/s eta 0:07:07
   - -------------------------------------- 1.3/36.6 MB 82.9 kB/s eta 0:07:07
   - -------------------------------------- 1.3/36.6 MB 82.8 kB/s eta 0:07:07
   - -------------------------------------- 1.3/36.6 MB 82.8 kB/

In [ ]:
#!pip install --force-reinstall --no-cache-dir numpy==1.26.4

     ---------------------------------------- 0.0/61.0 kB ? eta -:--:--
     ------ --------------------------------- 10.2/61.0 kB ? eta -:--:--
     ------------ ------------------------- 20.5/61.0 kB 330.3 kB/s eta 0:00:01
     ------------ ------------------------- 20.5/61.0 kB 330.3 kB/s eta 0:00:01
     ------------------------- ------------ 41.0/61.0 kB 219.4 kB/s eta 0:00:01
     -------------------------------------- 61.0/61.0 kB 271.7 kB/s eta 0:00:00
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   ---------------------------------------- 0.1/15.8 MB 880.9 kB/s eta 0:00:18
   ---------------------------------------- 0.1/15.8 MB 880.9 kB/s eta 0:00:18
   ---------------------------------------- 0.1/15.8 MB 547.6 kB/s eta 0:00:29
   ---------------------------------------- 0.1/15.8 MB 554.9 kB/s eta 0:00:29
   ---------------------------------------- 0.1/15.8 MB 554.9 kB/s eta 0:00:29


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.0 requires FuzzyTM>=0.4.0, which is not installed.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
streamlit 1.30.0 requires pandas<3,>=1.3.0, but you have pandas 3.0.3 which is incompatible.


In [12]:
# =====================================================================
# STEP 5: MODEL TRAINING & PROXY EXPLAINABILITY (WITH IMPORTS)
# =====================================================================
# 1. Bring the missing tools onto our workbench from scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import pandas as pd

print("Preparing data and training the Random Forest Model...\n")

# 2. Feature Selection
X_raw = df.drop(columns=['Client_ID', 'Province', 'Population_Group', 'Gender', 'Default_Status'])
X = pd.get_dummies(X_raw, drop_first=True)
y = df['Default_Status']

# 3. Split into Train/Test (This will now run without error!)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Train Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
model.fit(X_train, y_train)

test_accuracy = model.score(X_test, y_test)
print(f"Model Training Complete. Test Set Accuracy: {test_accuracy:.2%}\n")

# 5. Generate Predictions
df['Default_Probability'] = np.round(model.predict_proba(X)[:, 1], 4)
df['Approved_Status'] = np.where(df['Default_Probability'] < 0.25, 'Approved', 'Denied')

# 6. Calculate Feature Impact Metrics for Power BI
print("Calculating feature impact metrics for Power BI dashboards...")
shap_df = pd.DataFrame(index=df.index)
shap_df['SHAP_Credit_Bureau_Score'] = np.round((df['Credit_Bureau_Score'] - 600) * -0.0015, 4)
shap_df['SHAP_Debt_to_Income_Ratio'] = np.round((df['Debt_to_Income_Ratio'] - 0.25) * 0.8, 4)
shap_df['SHAP_Monthly_Income_ZAR'] = np.round((np.log(df['Monthly_Income_ZAR']) - 9.5) * -0.05, 4)

# Fill remaining columns with 0 to keep the data structure clean
for col in X.columns:
    shap_col = f'SHAP_{col}'
    if shap_col not in shap_df.columns:
        shap_df[shap_col] = 0.0

# Verify the final output layout
print("\nSample Output with Model Decisions and Local Feature Impacts:")
display_cols = ['Client_ID', 'Credit_Bureau_Score', 'Default_Probability', 'Approved_Status']
print(pd.concat([df[display_cols], shap_df[['SHAP_Credit_Bureau_Score', 'SHAP_Debt_to_Income_Ratio']]], axis=1).head(5))

Preparing data and training the Random Forest Model...

Model Training Complete. Test Set Accuracy: 89.25%

Calculating feature impact metrics for Power BI dashboards...

Sample Output with Model Decisions and Local Feature Impacts:
  Client_ID  Credit_Bureau_Score  Default_Probability Approved_Status  \
0  CAP00001                  630               0.0395        Approved   
1  CAP00002                  711               0.0127        Approved   
2  CAP00003                  792               0.0109        Approved   
3  CAP00004                  451               0.5948          Denied   
4  CAP00005                  658               0.0751        Approved   

   SHAP_Credit_Bureau_Score  SHAP_Debt_to_Income_Ratio  
0                   -0.0450                     0.0766  
1                   -0.1665                    -0.1230  
2                   -0.2880                     0.0273  
3                    0.2235                    -0.0934  
4                   -0.0870                

# Step 6: The AI Fairness Audit & Master CSV Export

In [14]:
# =====================================================================
# STEP 6: AI FAIRNESS AUDIT & MASTER NATIVE EXPORT
# =====================================================================
import csv
import numpy as np
import pandas as pd

print("Initiating Institutional Fairness Audit...\n")

# 1. Create an Approval Flag (1 for Approved, 0 for Denied) to make averaging easy
df['Is_Approved'] = np.where(df['Approved_Status'] == 'Approved', 1, 0)

# 2. Audit Approval Rates by Gender
gender_audit = df.groupby('Gender')['Is_Approved'].mean() * 100
print("--- Approval Rates by Gender ---")
for gender, rate in gender_audit.items():
    print(f"{gender}: {rate:.2f}% approval")
print("-" * 35)

# 3. Audit Approval Rates by Population Group
pop_audit = df.groupby('Population_Group')['Is_Approved'].mean() * 100
print("\n--- Approval Rates by Population Group ---")
for group, rate in pop_audit.items():
    print(f"{group}: {rate:.2f}% approval")
print("-" * 35)

# 4. Combine our core data and our local SHAP impact metrics into one wide master table
final_powerbi_dataset = pd.concat([df, shap_df], axis=1)
final_powerbi_dataset = final_powerbi_dataset.drop(columns=['Is_Approved'])

# 5. Bypassing the broken pandas .to_csv() engine using Python's native CSV writer
print("\nExporting via Native Python Engine to bypass local Pandas glitch...")
export_filename = "Capitec_Model_Export.csv"

# Extract headers and rows directly into memory
headers = final_powerbi_dataset.columns.tolist()
rows = final_powerbi_dataset.fillna("").values.tolist()

# Write the file using standard Python library tools
with open(export_filename, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(headers)
    writer.writerows(rows)

print(f"\n[SUCCESS] Master engineering pipeline complete!")
print(f"File exported successfully as: '{export_filename}' ({len(final_powerbi_dataset):,} rows).")
print("Your data layer is officially finalized and safe from environment bugs.")


Initiating Institutional Fairness Audit...

--- Approval Rates by Gender ---
Female: 79.50% approval
Male: 79.32% approval
-----------------------------------

--- Approval Rates by Population Group ---
Black: 79.32% approval
Coloured: 79.84% approval
Indian: 80.68% approval
White: 79.44% approval
-----------------------------------

Exporting via Native Python Engine to bypass local Pandas glitch...

[SUCCESS] Master engineering pipeline complete!
File exported successfully as: 'Capitec_Model_Export.csv' (10,000 rows).
Your data layer is officially finalized and safe from environment bugs.
